# 35 · E6D half-Vdd disturb-delay 今日上机 notebook

目标：先确认器件健康和今天的 reference MW，再跑 `±2.5 V` 反向小电压 disturb，扫 disturb-to-read delay，看 disturb 造成的读出 shift 是否随 delay 弛豫。

推荐顺序：

1. `S1`：只读 baseline，确认扎针/漏电。
2. `E1`：reference RAWD，确认今天这个点在 `Vg=-1.0 V` 短窗口能读到 MW。
3. `E6D`：ERS 后加 `-2.5 V / 100 µs`，PGM 后加 `+2.5 V / 100 µs`，扫 delay。
4. 读回 CSV 并画图。

注意：`LIVE=False` 时只做 dry-run，不开 VISA、不加载 DLL、不输出硬件。真正上机前把 `LIVE=True`。

In [1]:
from __future__ import annotations

import os
import re
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path(r"D:\test\B1500")
SCRIPT = ROOT / "scripts" / "wgfmu_next_round_minimal.py"
PYTHON = ROOT / ".venv" / "Scripts" / "python.exe"

# ====== 上机前主要改这里 ======
DEVICE_ID = "L10W10_04"      # 当前 L10 fresh 点位
GEOMETRY = "L10W10"          # 当前器件几何
LIVE = True                   # 今天已进入真机上机模式
VISA_ADDR = "GPIB1::17::INSTR"

# E1 reference 设置
E1_REPS = 1                   # 时间足可改 3
E1_FULL_DELAYS = True
E1_WIDE_VG = True

# E6D disturb 设置：第一轮建议只做 2.5V
E6D_REPS = 3
E6D_AMPS = "2.5"
E6D_DELAYS = "1e-6,1e-5,1e-4,1e-3,1e-2,1e-1,1"
E6D_IG_STOP_UA = 30

os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["B1500_VISA_ADDR"] = VISA_ADDR

print("ROOT", ROOT)
print("SCRIPT", SCRIPT)
print("LIVE", LIVE)
print("DEVICE_ID", DEVICE_ID, "GEOMETRY", GEOMETRY)

ROOT D:\test\B1500
SCRIPT D:\test\B1500\scripts\wgfmu_next_round_minimal.py
LIVE True
DEVICE_ID L10W10_04 GEOMETRY L10W10


In [2]:
def run_stage(stage: str, *extra: str) -> Path | None:
    """Run one stop-gated CLI stage and return OUTPUT_CSV path if present."""
    cmd = [str(PYTHON), str(SCRIPT), "--stage", stage, "--device-id", DEVICE_ID, "--geometry", GEOMETRY]
    if LIVE:
        cmd += ["--live", "--confirm", stage]
    cmd += list(extra)
    print("\n$", " ".join(cmd))
    proc = subprocess.run(cmd, cwd=str(ROOT), text=True, encoding="utf-8", errors="replace", capture_output=True, env=os.environ)
    print(proc.stdout)
    if proc.stderr:
        print("[stderr]")
        print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"stage {stage} failed with exit code {proc.returncode}")
    m = re.search(r"OUTPUT_CSV:\s*(.+)", proc.stdout)
    return Path(m.group(1).strip()) if m else None


def latest_csv(pattern: str) -> Path:
    files = sorted((ROOT / "runs" / ("live" if LIVE else "dry")).glob(pattern), key=lambda p: p.stat().st_mtime)
    if not files:
        raise FileNotFoundError(pattern)
    return files[-1]

## 1. S1 只读 baseline

先确认接触/漏电。若输出里 `REPORT_CODE` 不是 `S1_DONE_PROCEED_TO_E1`，或 `max_abs_Ig_A` 明显超过 `5 µA`，先不要继续 E1/E6D。

In [3]:
s1_csv = run_stage("S1")
s1_csv


$ D:\test\B1500\.venv\Scripts\python.exe D:\test\B1500\scripts\wgfmu_next_round_minimal.py --stage S1 --device-id L10W10_04 --geometry L10W10 --live --confirm S1
PLAN_BEGIN
live=True stage=S1 device_id=L10W10_04 geometry=L10W10
channels: Gate=202, Drain=201; allowed=[201, 202, 301] forbidden=[302]
stage_registry:
  S0: S0_open_fixture_smoke — open/fixture read-only smoke
  S1: S1_device_read_only_baseline — device read-only baseline
  E1: E1_RAWD_QUICK300ms_v2 — RAWD delay experiment
  E2: E2_minimal_A1_A100_C1_C10 — minimal read-disturb
  E3W: E3_pulse_width_scan — pulse-width scan
  E3A: E3_amplitude_scan — amplitude scan
  E4: E4_prebias — pre-bias polarity test
  E5: E5_visibility_grid — Vg/Vd read-window grid
  E6D: E6D_halfVdd_disturb_delay — half-Vdd disturb-delay
  CYCLE: CYCLE_checkpoint_endurance — checkpointed cycle endurance
S0: reps=5, no write, VG=[-0.2, 0.0, 0.2], VD=0.05 V, stop |Ig|>5 uA
S1: reps=20, no write, VG=[-0.2, 0.0, 0.2], VD=0.05 V, stop |Ig|>5 uA
E1: delays=

WindowsPath('D:/test/B1500/runs/live/20260528_212742_S1_device_read_only_baseline_L10W10_04/s1_device_read_only_baseline.csv')

## 2. E1 reference：确认今天这个点有短窗口 MW

默认使用 wide Vg + full delay。时间紧 `E1_REPS=1`；想稳一点改成 3。

In [4]:
e1_args = ["--e1-reps", str(E1_REPS)]
if E1_WIDE_VG:
    e1_args.append("--e1-wide-vg")
if E1_FULL_DELAYS:
    e1_args.append("--e1-full-delays")

e1_csv = run_stage("E1", *e1_args)
e1_csv


$ D:\test\B1500\.venv\Scripts\python.exe D:\test\B1500\scripts\wgfmu_next_round_minimal.py --stage E1 --device-id L10W10_04 --geometry L10W10 --live --confirm E1 --e1-reps 1 --e1-wide-vg --e1-full-delays
PLAN_BEGIN
live=True stage=E1 device_id=L10W10_04 geometry=L10W10
channels: Gate=202, Drain=201; allowed=[201, 202, 301] forbidden=[302]
stage_registry:
  S0: S0_open_fixture_smoke — open/fixture read-only smoke
  S1: S1_device_read_only_baseline — device read-only baseline
  E1: E1_RAWD_QUICK300ms_v2 — RAWD delay experiment
  E2: E2_minimal_A1_A100_C1_C10 — minimal read-disturb
  E3W: E3_pulse_width_scan — pulse-width scan
  E3A: E3_amplitude_scan — amplitude scan
  E4: E4_prebias — pre-bias polarity test
  E5: E5_visibility_grid — Vg/Vd read-window grid
  E6D: E6D_halfVdd_disturb_delay — half-Vdd disturb-delay
  CYCLE: CYCLE_checkpoint_endurance — checkpointed cycle endurance
S0: reps=5, no write, VG=[-0.2, 0.0, 0.2], VD=0.05 V, stop |Ig|>5 uA
S1: reps=20, no write, VG=[-0.2, 0.0, 0

RuntimeError: stage E1 failed with exit code 1

In [ ]:
def load_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    for col in ["delay_s", "Vg_read_V", "Vd_read_V", "Id_mean_A", "Ig_mean_A", "V_disturb_V", "delay_after_disturb_s"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def compute_mw(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    g = df.groupby(group_cols + ["state_target"], dropna=False)["Id_mean_A"].mean().unstack("state_target")
    if "ERS" in g.columns and "PGM" in g.columns:
        g["MW_A"] = g["ERS"] - g["PGM"]
    return g.reset_index()

e1 = load_csv(e1_csv)
e1.head()

In [ ]:
e1_mw = compute_mw(e1, ["delay_s", "Vg_read_V"])
display(e1_mw.sort_values(["Vg_read_V", "delay_s"]))

fig, ax = plt.subplots(figsize=(7, 4.5))
for vg, sub in e1_mw.groupby("Vg_read_V"):
    sub = sub.sort_values("delay_s")
    ax.semilogx(sub["delay_s"], sub["MW_A"] * 1e6, marker="o", label=f"Vg={vg:g} V")
ax.axhline(0, color="0.5", lw=1)
ax.set_xlabel("write-to-read delay (s)")
ax.set_ylabel("MW current = Id_ERS - Id_PGM (µA)")
ax.set_title("E1 reference MW vs delay")
ax.legend()
ax.grid(True, which="both", alpha=0.25)
plt.tight_layout()

fig_path = e1_csv.parent / "e1_reference_mw_vs_delay.png"
fig.savefig(fig_path, dpi=200)
fig_path

## 3. E6D half-Vdd disturb-delay

第一轮参数：

- ERS 初态后加 `-2.5 V / 100 µs` 反向 disturb。
- PGM 初态后加 `+2.5 V / 100 µs` 反向 disturb。
- disturb-to-read delay：`1 µs → 1 s`。
- 读点：`Vg=-1.0, -0.7, -0.4 V`。

判据：如果短 delay shift 大、长 delay shift 小，说明小电压 disturb 的影响会弛豫，支持可恢复 `Qocc / trap occupation`。

In [ ]:
e6d_csv = run_stage(
    "E6D",
    "--e6d-reps", str(E6D_REPS),
    "--e6d-amps", E6D_AMPS,
    "--e6d-delays", E6D_DELAYS,
    "--e6d-ig-stop-uA", str(E6D_IG_STOP_UA),
)
e6d_csv

In [ ]:
e6d = load_csv(e6d_csv)
display(e6d.head())

# Disturbed-state ERS/PGM difference after half-Vdd disturb.
e6d_mw = compute_mw(e6d, ["delay_after_disturb_s", "Vg_read_V"])
display(e6d_mw.sort_values(["Vg_read_V", "delay_after_disturb_s"]))

fig, ax = plt.subplots(figsize=(7, 4.5))
for vg, sub in e6d_mw.groupby("Vg_read_V"):
    sub = sub.sort_values("delay_after_disturb_s")
    ax.semilogx(sub["delay_after_disturb_s"], sub["MW_A"] * 1e6, marker="o", label=f"Vg={vg:g} V")
ax.axhline(0, color="0.5", lw=1)
ax.set_xlabel("disturb-to-read delay (s)")
ax.set_ylabel("disturbed MW current (µA)")
ax.set_title("E6D half-Vdd disturbed MW vs delay")
ax.legend()
ax.grid(True, which="both", alpha=0.25)
plt.tight_layout()

fig_path = e6d_csv.parent / "e6d_disturbed_mw_vs_delay.png"
fig.savefig(fig_path, dpi=200)
fig_path

In [ ]:
# 分初态看 Id 是否随 disturb-to-read delay 弛豫。
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, state in zip(axes, ["ERS", "PGM"]):
    sub0 = e6d[e6d["state_target"] == state]
    for vg, sub in sub0.groupby("Vg_read_V"):
        g = sub.groupby("delay_after_disturb_s")["Id_mean_A"].mean().reset_index().sort_values("delay_after_disturb_s")
        ax.semilogx(g["delay_after_disturb_s"], g["Id_mean_A"] * 1e6, marker="o", label=f"Vg={vg:g} V")
    ax.set_title(f"{state} initial + opposite 2.5V disturb")
    ax.set_xlabel("disturb-to-read delay (s)")
    ax.grid(True, which="both", alpha=0.25)
axes[0].set_ylabel("Id after disturb (µA)")
axes[1].legend(loc="best")
plt.tight_layout()

fig_path = e6d_csv.parent / "e6d_id_by_initial_state_vs_delay.png"
fig.savefig(fig_path, dpi=200)
fig_path

## 4. 快速判读模板

看完图后按这几条判断：

- `1–10 µs` 强、`100 ms–1 s` 弱：disturb 影响会弛豫，支持可恢复 trap/Qocc。
- 所有 delay 都差不多：可能是 partial switching / 不可逆 trap / 慢态，下一轮需要扫 `1.5/2.0/2.5/3.0 V`。
- 只有 `Vg=-1.0 V` 明显：读出投影 `Svis` 很关键。
- ERS 后 `-2.5V` 和 PGM 后 `+2.5V` 不对称：极性相关 trap / field asymmetry，是模型参数的重要约束。

把本 notebook 输出的 `OUTPUT_CSV` 和两张图发回给我，我可以继续做正式判读和归档。